In [1]:
# !pip install requests geopandas
# !pip install rasterio
# !conda install -c conda-forge gdal -y


In [2]:
import requests
import geopandas as gpd
from shapely.geometry import shape


import rasterio
from rasterio.io import MemoryFile
from shapely.geometry import box
import geopandas as gpd
import os
from rasterio.transform import from_bounds
import glob
import pandas as pd
from datetime import datetime
import time
from osgeo import gdal

import logging

In [3]:
# Get the GDAL library path
gdal_path = os.path.dirname(gdal.__file__)
print(f"GDAL is installed at: {gdal_path}")


GDAL is installed at: C:\Users\Biplov.Parajuli\AppData\Local\anaconda3\envs\Geodata_Bundeslander\Lib\site-packages\osgeo


# SAARLAND

## Downloading parcels from OGC API

In [ ]:
import requests
import geopandas as gpd
from shapely.geometry import shape

# for i in range(0,1281000,500):
for i in range(1281000,1282000,500):
    
 
    try:
 
        url = "https://geoportal.saarland.de/spatial-objects/408/collections/cp:CadastralParcel/items?offset={}&limit=500&f=json".format(i)
        
        # Make the GET request to the API
        response = requests.get(url)
        # Check the response status code and content
        print(f"Status Code: {response.status_code}")
        print("Response Text:")
        print(response.text[:500])  # Print the first 500 characters of the response text
        # Step 4: Retrieve and Parse Data
        # Check if the response is in GeoJSON format
        try:
            data = response.json()
            print("Data successfully parsed as JSON.")
        except requests.exceptions.JSONDecodeError:
            print("Failed to parse response as JSON.")
            data = None
            # Verify if data is GeoJSON by checking for 'type' and 'features' keys
            if 'type' in data and data['type'] == 'FeatureCollection' and 'features' in data:

                # Extract features from the data
                features = data['features']
                
                # Convert features to GeoDataFrame
                geometries = [shape(feature['geometry']) for feature in features]
                properties = [feature['properties'] for feature in features]
                gdf = gpd.GeoDataFrame(properties, geometry=geometries)
               
                # Set the CRS for the GeoDataFrame
                gdf.set_crs(epsg=4326, inplace=True)

                # Step 5: Save the Data
                # Save the GeoDataFrame to a GeoPackage
                output = r"g:\0 BIPLOV\Reference GeoData Preparation_Bundesland\ALKIS\a\cadastral_parcels_saarland_{}.gpkg".format(i)
                gdf.to_file(output, layer='cadastral_parcels_{}'.format(i), driver="GPKG")
                print("done {}".format(i))
            else:
                pass
        else:
            pass
    except:
        pass


## Download Orthophoto from the grid

In [ ]:
# # Step 1: Install Necessary Libraries
# !pip install requests rasterio shapely geopandas

# # Step 2: Import Libraries
# import requests
# import rasterio
# from rasterio.io import MemoryFile
# from shapely.geometry import box
# import geopandas as gpd
# import os

# Step 3: Define the grid parameters
origin_x = 2522682.4534885  # Starting x-coordinate
origin_y = 5440357.0644289  # Starting y-coordinate
cell_width = 2000  # Width of each cell in meters
cell_height = 2000  # Height of each cell in meters
num_columns = 31  # Number of columns in the grid
num_rows = 40  # Number of rows in the grid
output_dir = r"T:\0 BIPLOV\Reference GeoData Preparation_Bundesland\Saarland\Orthophoto" # Directory to save images

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Step 4: Generate the grid of bounding boxes
grid_cells = []
for i in range(num_columns):
    for j in range(num_rows):
        minx = origin_x + i * cell_width
        maxx = minx + cell_width
        miny = origin_y + j * cell_height
        maxy = miny + cell_height
        grid_cells.append((minx, miny, maxx, maxy))

# Step 5: Function to download WMS data for a given bounding box
def download_wms_image(bbox, filename):
    base_url = ("https://geoportal.lkvk.saarland.de/freewms/dop2006"
                "?REQUEST=GetMap"
                "&VERSION=1.1.1"
                "&SERVICE=WMS"
                "&LAYERS=Orthophoto%202006"
                "&STYLES="
                "&SRS=EPSG:31466"
                "&BBOX={},{},{},{}"
                "&WIDTH=5000"
                "&HEIGHT=5000"
                "&FORMAT=image/tiff"
                "&BGCOLOR=0xffffff"
                "&TRANSPARENT=TRUE"
                "&EXCEPTIONS=application/vnd.ogc.se_inimage")
    
    url = base_url.format(bbox[0], bbox[1], bbox[2], bbox[3])
    
    response = requests.get(url)
    
    if response.status_code == 200:
        with MemoryFile(response.content) as memfile:
            with memfile.open() as dataset:
                profile = dataset.profile.copy()
                profile.update({
                    'driver': 'GTiff',
                    'height': dataset.height,
                    'width': dataset.width,
                    'count': dataset.count,
                    'dtype': dataset.dtypes[0],
                    'crs': dataset.crs,
                    'transform': from_bounds(bbox[0], bbox[1], bbox[2], bbox[3], dataset.width, dataset.height)
                })
                with rasterio.open(filename, 'w', **profile) as dst:
                    dst.write(dataset.read())
        print(f"Image saved to {filename}")
    else:
        print(f"Failed to download image for bbox {bbox}: {response.status_code}")

# Step 6: Download images for each grid cell
for index, bbox in enumerate(grid_cells):
    filename = os.path.join(output_dir, f"Orthophoto_2006_{index+1}.tiff")
    download_wms_image(bbox, filename)



# HESSEN

## Downloading parcels from OGC API

### With Delay

In [ ]:
# Define the input directory
input_dir = r"D:\BIPLOV\Reference GeoData Preparation_Bundesland\HESSEN\Flurstueck"
output_file = os.path.join(input_dir, 'Flurstueck_Hessen.gpkg')

# Record the initial time
initial_time = datetime.now()
print(f"Initial time: {initial_time}")

# Function to download and save GeoPackage files
def download_geopackage_files(start_offset, end_offset, step, max_retries=5, base_delay=5):
    for i in range(start_offset, end_offset, step):
        retries = 0
        success = False
        while retries < max_retries and not success:
            try:
                url = f"https://www.geoportal.hessen.de/spatial-objects/710/collections/cp:CadastralParcel/items?offset={i}&limit=500&f=json"
                response = requests.get(url)
                print(f"Status Code for offset {i}: {response.status_code}")
                
                if response.status_code == 502:
                    retries += 1
                    delay = base_delay * (2 ** (retries - 1))
                    print(f"Retrying {retries}/{max_retries} after status 502 for offset {i} in {delay} seconds...")
                    time.sleep(delay)
                    continue

                response.raise_for_status()

                try:
                    data = response.json()
                    print(f"Data successfully parsed as JSON for offset {i}.")
                except requests.exceptions.JSONDecodeError:
                    print(f"Failed to parse response as JSON for offset {i}.")
                    data = None
                
                if data and 'type' in data and data['type'] == 'FeatureCollection' and 'features' in data:
                    features = data['features']
                    geometries = [shape(feature['geometry']) for feature in features]
                    properties = [feature['properties'] for feature in features]
                    gdf = gpd.GeoDataFrame(properties, geometry=geometries)
                    gdf.set_crs(epsg=4326, inplace=True)
                    output = os.path.join(input_dir, f'cadastral_parcels_hessen_{i}.gpkg')
                    gdf.to_file(output, layer=f'cadastral_parcels_{i}', driver="GPKG")
                    print(f"Done {i}")
                    success = True
                else:
                    print(f"No valid data found for offset {i}.")
                    break

            except requests.RequestException as e:
                print(f"Error occurred for offset {i}: {e}")
                retries += 1
                delay = base_delay * (2 ** (retries - 1))
                time.sleep(delay)
        
        if not success:
            print(f"Max retries reached for offset {i}. Skipping...")

# Call the function to download and save GeoPackage files
download_geopackage_files(2733500, 4990000, 500)

# Record the end time
end_time = datetime.now()
print(f"End time: {end_time}")

# Calculate and print the duration
duration = end_time - initial_time
print(f"Duration: {duration}")

Initial time: 2024-08-13 12:05:18.400958
Status Code for offset 2733500: 502
Retrying 1/5 after status 502 for offset 2733500 in 5 seconds...
Status Code for offset 2733500: 502
Retrying 2/5 after status 502 for offset 2733500 in 10 seconds...
Status Code for offset 2733500: 502
Retrying 3/5 after status 502 for offset 2733500 in 20 seconds...
Status Code for offset 2733500: 502
Retrying 4/5 after status 502 for offset 2733500 in 40 seconds...
Status Code for offset 2733500: 502
Retrying 5/5 after status 502 for offset 2733500 in 80 seconds...
Max retries reached for offset 2733500. Skipping...
Status Code for offset 2734000: 502
Retrying 1/5 after status 502 for offset 2734000 in 5 seconds...
Status Code for offset 2734000: 502
Retrying 2/5 after status 502 for offset 2734000 in 10 seconds...
Status Code for offset 2734000: 502
Retrying 3/5 after status 502 for offset 2734000 in 20 seconds...
Status Code for offset 2734000: 502
Retrying 4/5 after status 502 for offset 2734000 in 40 se

### With Delay 1

In [ ]:
# Configure logging
logging.basicConfig(filename='download_geopackage.log', level=logging.INFO, format='%(asctime)s %(message)s')

# Define the input directory
input_dir = r"D:\BIPLOV\Reference GeoData Preparation_Bundesland\HESSEN\Flurstueck"
output_file = os.path.join(input_dir, 'Flurstueck_Hessen.gpkg')

# Record the initial time
initial_time = datetime.now()
print(f"Initial time: {initial_time}")

# Function to download and save GeoPackage files
def download_geopackage_files(start_offset, end_offset, step, max_retries=10, base_delay=10):
    for i in range(start_offset, end_offset, step):
        retries = 0
        success = False
        while retries < max_retries and not success:
            try:
                url = f"https://www.geoportal.hessen.de/spatial-objects/710/collections/cp:CadastralParcel/items?offset={i}&limit=500&f=json"
                response = requests.get(url)
                logging.info(f"Status Code for offset {i}: {response.status_code}")
                
                if response.status_code == 502:
                    retries += 1
                    delay = base_delay * (2 ** (retries - 1))
                    logging.info(f"Retrying {retries}/{max_retries} after status 502 for offset {i} in {delay} seconds...")
                    time.sleep(delay)
                    continue

                response.raise_for_status()

                try:
                    data = response.json()
                    logging.info(f"Data successfully parsed as JSON for offset {i}.")
                except requests.exceptions.JSONDecodeError:
                    logging.error(f"Failed to parse response as JSON for offset {i}.")
                    data = None
                
                if data and 'type' in data and data['type'] == 'FeatureCollection' and 'features' in data:
                    features = data['features']
                    geometries = [shape(feature['geometry']) for feature in features]
                    properties = [feature['properties'] for feature in features]
                    gdf = gpd.GeoDataFrame(properties, geometry=geometries)
                    gdf.set_crs(epsg=4326, inplace=True)
                    output = os.path.join(input_dir, f'cadastral_parcels_hessen_{i}.gpkg')
                    gdf.to_file(output, layer=f'cadastral_parcels_{i}', driver="GPKG")
                    logging.info(f"Done {i}")
                    success = True
                else:
                    logging.warning(f"No valid data found for offset {i}.")
                    break

            except requests.RequestException as e:
                logging.error(f"Error occurred for offset {i}: {e}")
                retries += 1
                delay = base_delay * (2 ** (retries - 1))
                time.sleep(delay)
        
        if not success:
            logging.error(f"Max retries reached for offset {i}. Skipping...")

# Call the function to download and save GeoPackage files
download_geopackage_files(2733500, 4990000, 500)

# Record the end time
end_time = datetime.now()
print(f"End time: {end_time}")

# Calculate and print the duration
duration = end_time - initial_time
print(f"Duration: {duration}")


### Without delay

In [ ]:
# Define the input directory
input_dir = r"D:\BIPLOV\Reference GeoData Preparation_Bundesland\HESSEN\Flurstueck"
output_file = os.path.join(input_dir, 'Flurstueck_Hessen.gpkg')

# Record the initial time
initial_time = datetime.now()
print(f"Initial time: {initial_time}")

# Function to download and save GeoPackage files
def download_geopackage_files(start_offset, end_offset, step, max_retries=10):
    for i in range(start_offset, end_offset, step):
        retries = 0
        success = False
        while retries < max_retries and not success:
            try:
                url = f"https://www.geoportal.hessen.de/spatial-objects/710/collections/cp:CadastralParcel/items?offset={i}&limit=500&f=json"
                response = requests.get(url)
                print(f"Status Code for offset {i}: {response.status_code}")
                
                if response.status_code == 502:
                    retries += 1
                    print(f"Retrying {retries}/{max_retries} after status 502 for offset {i}...")
                    continue

                response.raise_for_status()

                try:
                    data = response.json()
                    print(f"Data successfully parsed as JSON for offset {i}.")
                except requests.exceptions.JSONDecodeError:
                    print(f"Failed to parse response as JSON for offset {i}.")
                    data = None
                
                if data and 'type' in data and data['type'] == 'FeatureCollection' and 'features' in data:
                    features = data['features']
                    geometries = [shape(feature['geometry']) for feature in features]
                    properties = [feature['properties'] for feature in features]
                    gdf = gpd.GeoDataFrame(properties, geometry=geometries)
                    gdf.set_crs(epsg=4326, inplace=True)
                    output = os.path.join(input_dir, f'cadastral_parcels_hessen_{i}.gpkg')
                    gdf.to_file(output, layer=f'cadastral_parcels_{i}', driver="GPKG")
                    print(f"Done {i}")
                    success = True
                else:
                    print(f"No valid data found for offset {i}.")
                    break

            except requests.RequestException as e:
                print(f"Error occurred for offset {i}: {e}")
                retries += 1
        
        if not success:
            print(f"Max retries reached for offset {i}. Skipping...")

# Call the function to download and save GeoPackage files
download_geopackage_files(2733500, 4990000, 500)

# Record the end time
end_time = datetime.now()
print(f"End time: {end_time}")

# Calculate and print the duration
duration = end_time - initial_time
print(f"Duration: {duration}")